In [1]:
import re
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, confusion_matrix, classification_report


In [2]:
HTML_JUNK = re.compile(
	r"\b(div|span|p|br|strong|b|i|em|ul|ol|li|table|tr|td|th|thead|tbody|"
	r"class|id|style|align|width|height|border|cellpadding|cellspacing)\b",
	flags=re.IGNORECASE
)

def light_html_strip(text: str) -> str:
	if not isinstance(text, str):
		return ""
	text = HTML_JUNK.sub(" ", text)
	text = re.sub(r"\s+", " ", text)
	return text.strip()


In [3]:
def preprocess_article_light(X: pd.Series) -> pd.Series:
	return (
		X.fillna("")
		 .astype(str)
		 .map(light_html_strip)
	)


In [4]:
df = pd.read_csv("../data/processed/v3/development_v3.csv")

X_df = df[["title", "article", "source"]].copy()
y = df["label"].values


In [5]:
title_pipe = Pipeline([
	("sel", FunctionTransformer(lambda d: d["title"].fillna("").astype(str), validate=False)),
	("tfidf", TfidfVectorizer(
		ngram_range=(1,2),
		min_df=2,
		max_df=0.9,
		sublinear_tf=True,
		max_features=250_000
	))
])


In [6]:
article_word_pipe = Pipeline([
	("sel", FunctionTransformer(lambda d: preprocess_article_light(d["article"]), validate=False)),
	("tfidf", TfidfVectorizer(
		ngram_range=(1,2),
		min_df=2,
		max_df=0.9,
		sublinear_tf=True,
		max_features=350_000
	))
])


In [7]:
article_char_pipe = Pipeline([
	("sel", FunctionTransformer(lambda d: preprocess_article_light(d["article"]), validate=False)),
	("tfidf", TfidfVectorizer(
		analyzer="char_wb",
		ngram_range=(3,5),
		min_df=3,
		max_df=0.9,
		sublinear_tf=True,
		max_features=300_000
	))
])


In [8]:
source_pipe = Pipeline([
	("sel", FunctionTransformer(lambda d: d["source"].fillna("").astype(str), validate=False)),
	("tfidf", TfidfVectorizer(
		ngram_range=(1,2),
		min_df=2,
		max_features=50_000
	))
])


In [9]:
model = Pipeline([
	("features", FeatureUnion([
		("title", title_pipe),
		("article_word", article_word_pipe),
		("article_char", article_char_pipe),
		("source", source_pipe),
	])),
	("clf", LogisticRegression(
		C=2.0,
		max_iter=2000,
		n_jobs=-1,
		class_weight="balanced"
	))
])


In [10]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for tr, te in skf.split(X_df, y):
	model.fit(X_df.iloc[tr], y[tr])
	yp = model.predict(X_df.iloc[te])

	print("Macro F1:", f1_score(y[te], yp, average="macro"))
	print("Macro Recall:", recall_score(y[te], yp, average="macro"))
	print("Confusion Matrix:\n", confusion_matrix(y[te], yp))
	print("\nClassification Report:\n",
		  classification_report(y[te], yp, digits=3))
	break


Macro F1: 0.7091403962446646
Macro Recall: 0.7247399372556391
Confusion Matrix:
 [[3415  146  118  263   45  639   82]
 [  83 1698  107   92   17   80   40]
 [  80  124 1844   75    8   53   48]
 [ 229  103  108 1112  121  256   67]
 [  22    8    2   68 1559   50    6]
 [ 591  127   56  313  123 1326   75]
 [  42   17   16   34   13   36  463]]

Classification Report:
               precision    recall  f1-score   support

           0      0.765     0.725     0.745      4708
           1      0.764     0.802     0.782      2117
           2      0.819     0.826     0.823      2232
           3      0.568     0.557     0.563      1996
           4      0.827     0.909     0.866      1715
           5      0.543     0.508     0.525      2611
           6      0.593     0.746     0.660       621

    accuracy                          0.714     16000
   macro avg      0.697     0.725     0.709     16000
weighted avg      0.712     0.714     0.712     16000



In [11]:
def make_text(row):
	title = row["title"] if isinstance(row["title"], str) else ""
	article = row["article"] if isinstance(row["article"], str) else ""

	title = light_html_strip(title)
	article = light_html_strip(article)

	# BOOST TITLE x3
	return f"{title} {title} {title} {article}"


In [12]:
df = pd.read_csv("../data/processed/v3/development_v3.csv")

X = df.apply(make_text, axis=1)
y = df["label"].values


In [13]:
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

model = Pipeline([
	("tfidf", TfidfVectorizer(
		analyzer="char",
		ngram_range=(3,3),
		min_df=3,
		max_df=0.95,
		sublinear_tf=True,
		max_features=400_000
	)),
	("clf", SGDClassifier(
		loss="log_loss",
		alpha=1e-5,
		penalty="l2",
		max_iter=1000,
		n_jobs=-1,
		random_state=42
	))
])


In [14]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, confusion_matrix

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for tr, te in skf.split(X, y):
	model.fit(X.iloc[tr], y[tr])
	yp = model.predict(X.iloc[te])

	print("Macro F1:", f1_score(y[te], yp, average="macro"))
	print("Macro Recall:", recall_score(y[te], yp, average="macro"))
	print("Confusion Matrix:\n", confusion_matrix(y[te], yp))
	break


Macro F1: 0.6602163365719196
Macro Recall: 0.6643868334207556
Confusion Matrix:
 [[3714  176  158  153   89  369   49]
 [ 170 1517  188   56   16  142   28]
 [ 165  147 1732   57   29   59   43]
 [ 429  130  130  947  143  167   50]
 [  90   12   10   50 1511   36    6]
 [ 861  184  135  196  164 1003   68]
 [  93   22   28   35   13   39  391]]


In [15]:
from nltk.stem import PorterStemmer
import re

stemmer = PorterStemmer()

def dirty_preprocess(text):
	if not isinstance(text, str):
		return ""
	text = light_html_strip(text.lower())
	text = re.sub(r"[^a-z0-9 ]", " ", text)
	tokens = text.split()
	tokens = [stemmer.stem(t) for t in tokens if len(t) > 2]
	return " ".join(tokens)


In [16]:
def make_text(row):
	t = row["title"] if isinstance(row["title"], str) else ""
	a = row["article"] if isinstance(row["article"], str) else ""

	t = dirty_preprocess(t)
	a = dirty_preprocess(a)

	return f"{t} {t} {t} {a}"


In [17]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

model = Pipeline([
	("tfidf", TfidfVectorizer(
		analyzer="char",
		ngram_range=(3,3),
		min_df=3,
		max_df=0.95,
		sublinear_tf=True,
		max_features=300_000
	)),
	("clf", MultinomialNB(alpha=0.01))
])


In [18]:
import re

HTML_JUNK = re.compile(
	r"\b(div|span|p|br|strong|b|i|em|ul|ol|li|table|tr|td|th|thead|tbody|"
	r"class|id|style|align|width|height|border|cellpadding|cellspacing)\b",
	flags=re.IGNORECASE
)

def light_html_strip(text):
	if not isinstance(text, str):
		return ""
	text = HTML_JUNK.sub(" ", text.lower())
	text = re.sub(r"\s+", " ", text)
	return text.strip()


In [19]:
import re

HTML_JUNK = re.compile(
	r"\b(div|span|p|br|strong|b|i|em|ul|ol|li|table|tr|td|th|thead|tbody|"
	r"class|id|style|align|width|height|border|cellpadding|cellspacing)\b",
	flags=re.IGNORECASE
)

def light_html_strip(text):
	if not isinstance(text, str):
		return ""
	text = HTML_JUNK.sub(" ", text.lower())
	text = re.sub(r"\s+", " ", text)
	return text.strip()


In [20]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def dirty_preprocess(text):
	text = light_html_strip(text)
	text = re.sub(r"[^a-z0-9 ]", " ", text)
	tokens = text.split()
	tokens = [stemmer.stem(t) for t in tokens if len(t) > 2]
	return " ".join(tokens)


In [21]:
def make_text(row):
	t = row["title"] if isinstance(row["title"], str) else ""
	a = row["article"] if isinstance(row["article"], str) else ""

	t = dirty_preprocess(t)
	a = dirty_preprocess(a)

	return f"{t} {a}"


In [22]:
import pandas as pd

df = pd.read_csv("../data/processed/v3/development_v3.csv")

X = df.apply(make_text, axis=1)
y = df["label"].values


In [25]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

model = Pipeline([
	("tfidf", TfidfVectorizer(
		analyzer="char",
		ngram_range=(2,3),
		min_df=3,
		max_df=0.95,
		sublinear_tf=True,
		max_features=400_000
	)),
	("clf", SGDClassifier(
		loss="log_loss",
		alpha=1e-5,
		penalty="l2",
		max_iter=1000,
		n_jobs=-1,
		random_state=42
	))
])


In [26]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, confusion_matrix

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for tr, te in skf.split(X, y):
	model.fit(X.iloc[tr], y[tr])
	yp = model.predict(X.iloc[te])

	print("Macro F1:", f1_score(y[te], yp, average="macro"))
	print("Macro Recall:", recall_score(y[te], yp, average="macro"))
	print("Confusion Matrix:\n", confusion_matrix(y[te], yp))
	break


Macro F1: 0.6235849259465243
Macro Recall: 0.6312889034050715
Confusion Matrix:
 [[3658  215  172  160  105  330   68]
 [ 202 1450  224   59   11  131   40]
 [ 197  180 1625   78   35   63   54]
 [ 536  118  143  887  143  117   52]
 [ 106   10   13   54 1485   43    4]
 [1098  176  152  185  169  755   76]
 [ 112   25   37   17   14   25  391]]


In [28]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# testo semplice (article o title+article)
X_text = df["article"].fillna("").astype(str)
y = df["label"]

vec = TfidfVectorizer(
	stop_words="english",
	min_df=5,
	max_df=0.9,
	ngram_range=(1,1)
)

X_tfidf = vec.fit_transform(X_text)
terms = np.array(vec.get_feature_names_out())

top_words = {}

for c in sorted(y.unique()):
	idx = (y == c).values
	mean_tfidf = X_tfidf[idx].mean(axis=0).A1
	top_idx = mean_tfidf.argsort()[-20:][::-1]
	top_words[c] = terms[top_idx]

for c, w in top_words.items():
	print(f"\nClass {c}:")
	print(", ".join(w))



Class 0:
com, ap, http, afp, said, reuters, 130, yimg, yahoo, president, iraq, minister, world, img, rss, new, people, left, government, src

Class 1:
reuters, com, fool, http, usmf, foolwatch, new, said, feeds, img, york, company, percent, stocks, 39, oil, corp, billion, prices, fullquote

Class 2:
com, http, computerworld, pcworld, feeds, img, h4, new, news, src, wired, latestnews, border, href, microsoft, company, software, internet, said, ap

Class 3:
39, com, ap, reuters, http, quot, new, film, said, entertainment, yahoo, 130, yimg, angeles, los, img, year, hollywood, york, news

Class 4:
ap, 39, game, com, team, sports, season, victory, night, http, new, win, league, coach, points, second, sunday, 130, yimg, yahoo

Class 5:
cnn, com, http, reuters, rss, cnn_topstories, 39, img, www, said, president, src, iraq, href, newsisfree, sources, new, info, quot, washington

Class 6:
health, cancer, new, reuters, study, drug, researchers, healthday, disease, risk, flu, said, 39, children,

In [29]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report


In [31]:
df = pd.read_csv("../data/processed/v3/development_v3.csv")

df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("unknown").astype(str)

X_text = df["article"]
X_source = df["source"]
y = df["label"]

# 1 fold fisso
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr, te = next(skf.split(X_text, y))



In [32]:
model_base = Pipeline([
	("tfidf", TfidfVectorizer(
		stop_words="english",
		min_df=3,
		max_df=0.9,
		ngram_range=(1,2),
		sublinear_tf=True,
		max_features=150_000
	)),
	("clf", LogisticRegression(
		max_iter=1000,
		n_jobs=-1
	))
])

model_base.fit(X_text.iloc[tr], y.iloc[tr])
proba_base = model_base.predict_proba(X_text.iloc[te])
yp_base = np.argmax(proba_base, axis=1)

print("BASELINE F1:", f1_score(y.iloc[te], yp_base, average="macro"))


KeyboardInterrupt: 

In [33]:
# P(label | source)
source_counts = (
	df.iloc[tr]
	.groupby(["source", "label"])
	.size()
	.unstack(fill_value=0)
)

source_prior = source_counts.div(source_counts.sum(axis=1), axis=0)


In [34]:
alpha = 0.7  # peso del modello testuale

proba_blended = []

for i, idx in enumerate(te):
	src = X_source.iloc[idx]

	p_model = proba_base[i]

	if src in source_prior.index:
		p_src = source_prior.loc[src].values
		p_final = alpha * p_model + (1 - alpha) * p_src
	else:
		p_final = p_model

	proba_blended.append(p_final)

proba_blended = np.vstack(proba_blended)
yp_blend = np.argmax(proba_blended, axis=1)

print("BLENDED F1:", f1_score(y.iloc[te], yp_blend, average="macro"))


NameError: name 'proba_base' is not defined

In [ ]:
import numpy as np
import pandas as pd
import re

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("../data/processed/v3/development_v3.csv")

df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("unknown").astype(str)

X_text = df["article"]
y = df["label"]

# =========================
# 1 FOLD SPLIT
# =========================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr, te = next(skf.split(X_text, y))

# =========================
# BASE MODEL
# =========================
model = Pipeline([
	("tfidf", TfidfVectorizer(
		stop_words="english",
		min_df=3,
		max_df=0.9,
		ngram_range=(1,2),
		sublinear_tf=True,
		max_features=150_000
	)),
	("clf", LogisticRegression(
		max_iter=1000,
		n_jobs=-1
	))
])

model.fit(X_text.iloc[tr], y.iloc[tr])
proba = model.predict_proba(X_text.iloc[te])

# =========================
# KEYWORD RULES (HIGH PRECISION)
# =========================
KEYWORDS = {
	4: ["game", "team", "season", "league", "coach", "points", "win"],
	2: ["software", "microsoft", "internet", "pcworld", "wired", "computer"],
	6: ["cancer", "drug", "study", "vaccine", "disease", "patients"],
	1: ["stocks", "percent", "prices", "billion", "oil", "shares"]
}

def keyword_override(text):
	text = text.lower()
	for label, words in KEYWORDS.items():
		for w in words:
			if re.search(rf"\b{w}\b", text):
				return label
	return None

# =========================
# HYBRID DECISION
# =========================
tau = 0.45  # soglia incertezza
fallback = y.iloc[tr].value_counts().idxmax()

y_pred = []

for i, idx in enumerate(te):
	p = proba[i]
	text = X_text.iloc[idx]

	if p.max() >= tau:
		y_pred.append(p.argmax())
	else:
		override = keyword_override(text)
		if override is not None:
			y_pred.append(override)
		else:
			y_pred.append(fallback)

y_pred = np.array(y_pred)

# =========================
# METRICS
# =========================
print("HYBRID MACRO F1:", f1_score(y.iloc[te], y_pred, average="macro"))
print()
print(classification_report(y.iloc[te], y_pred))


In [35]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import re

df = pd.read_csv("../data/processed/v3/development_v3.csv")
df["article"] = df["article"].fillna("").astype(str)
df["label"] = df["label"].astype(int)

def tokenize(text):
	return re.findall(r"\b[a-z]{3,}\b", text.lower())

tokens = df["article"].map(tokenize)


In [37]:
word_class_counts = defaultdict(Counter)

for toks, label in zip(tokens, df["label"]):
	word_class_counts[label].update(toks)

# totale globale
global_counts = Counter()
for c in word_class_counts:
	global_counts.update(word_class_counts[c])



In [38]:
rows = []

for word, total_freq in global_counts.items():
	per_class = {c: word_class_counts[c][word] for c in word_class_counts}
	dominant_class = max(per_class, key=per_class.get)
	dom_freq = per_class[dominant_class]

	purity = dom_freq / total_freq if total_freq > 0 else 0

	rows.append({
		"word": word,
		"total_freq": total_freq,
		"dominant_class": dominant_class,
		"dominant_freq": dom_freq,
		"purity": purity,
		**{f"class_{c}_freq": per_class[c] for c in per_class}
	})

word_df = pd.DataFrame(rows)


In [39]:
pure_words = word_df[
	(word_df["total_freq"] >= 50) &
	(word_df["purity"] >= 0.85)
].sort_values(["purity", "total_freq"], ascending=False)


In [40]:
pure_words.groupby("dominant_class").head(20)


,word,total_freq,dominant_class,dominant_freq,purity,class_5_freq,class_0_freq,class_3_freq,class_2_freq,class_4_freq,class_1_freq,class_6_freq
46255,pcworld,7024,2,7024,1.000000,0,0,0,7024,0,0,0
46256,latestnews,3640,2,3640,1.000000,0,0,0,3640,0,0,0
46346,topheadlines,1958,2,1958,1.000000,0,0,0,1958,0,0,0
61235,usmf,1500,1,1500,1.000000,0,0,0,0,0,1500,0
61236,foolwatch,1500,1,1500,1.000000,0,0,0,0,0,1500,0
...,...,...,...,...,...,...,...,...,...,...,...,...
14699,warsaw,50,0,44,0.880000,2,44,2,0,0,2,0
2932,usa,1013,0,891,0.879566,18,891,26,22,37,16,3
11184,actors,116,3,102,0.879310,5,8,102,0,0,1,0
4188,kosovo,178,0,154,0.865169,21,154,3,0,0,0,0


In [41]:
ANCHOR_WORDS = {
	c: set(pure_words[pure_words["dominant_class"] == c]["word"])
	for c in pure_words["dominant_class"].unique()
}


In [42]:
def anchor_override(text):
	toks = set(tokenize(text))
	for label, words in ANCHOR_WORDS.items():
		if toks & words:
			return label
	return None


In [43]:
tau = 0.45
fallback = df["label"].value_counts().idxmax()

y_pred = []

for i, idx in enumerate(te):
	p = proba[i]
	text = X_text.iloc[idx]

	if p.max() >= tau:
		y_pred.append(p.argmax())
	else:
		override = anchor_override(text)
		if override is not None:
			y_pred.append(override)
		else:
			y_pred.append(fallback)


NameError: name 'proba' is not defined

In [44]:
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report


In [45]:
def tokenize(text):
	return re.findall(r"\b[a-z]{3,}\b", text.lower())

tokens = X_text.map(tokenize)


In [46]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr, te = next(skf.split(X_text, y))


In [47]:
word_class_counts = defaultdict(Counter)

for toks, label in zip(tokens.iloc[tr], y.iloc[tr]):
	word_class_counts[label].update(toks)

global_counts = Counter()
for c in word_class_counts:
	global_counts.update(word_class_counts[c])


In [48]:
rows = []

for word, total_freq in global_counts.items():
	per_class = {c: word_class_counts[c][word] for c in word_class_counts}
	dom_class = max(per_class, key=per_class.get)
	dom_freq = per_class[dom_class]
	purity = dom_freq / total_freq if total_freq > 0 else 0

	rows.append({
		"word": word,
		"total_freq": total_freq,
		"dominant_class": dom_class,
		"purity": purity
	})

word_df = pd.DataFrame(rows)


In [49]:
PURE_WORDS = word_df[
	(word_df["total_freq"] >= 50) &
	(word_df["purity"] >= 0.85)
]


In [50]:
ANCHOR_WORDS = {
	c: set(PURE_WORDS[PURE_WORDS["dominant_class"] == c]["word"])
	for c in PURE_WORDS["dominant_class"].unique()
}


In [51]:
model = Pipeline([
	("tfidf", TfidfVectorizer(
		stop_words="english",
		min_df=3,
		max_df=0.9,
		ngram_range=(1,2),
		sublinear_tf=True,
		max_features=150_000
	)),
	("clf", LogisticRegression(
		max_iter=1000,
		n_jobs=-1
	))
])

model.fit(X_text.iloc[tr], y.iloc[tr])
proba = model.predict_proba(X_text.iloc[te])


In [52]:
def anchor_override(text):
	toks = set(tokenize(text))
	for label, words in ANCHOR_WORDS.items():
		if toks & words:
			return label
	return None


In [53]:
tau = 0.45  # confidence threshold
fallback = y.iloc[tr].value_counts().idxmax()

y_pred = []

for i, idx in enumerate(te):
	p = proba[i]
	text = X_text.iloc[idx]

	if p.max() >= tau:
		y_pred.append(p.argmax())
	else:
		override = anchor_override(text)
		if override is not None:
			y_pred.append(override)
		else:
			y_pred.append(fallback)

y_pred = np.array(y_pred)


In [54]:
print("HYBRID MACRO F1:", f1_score(y.iloc[te], y_pred, average="macro"))
print()
print(classification_report(y.iloc[te], y_pred))


HYBRID MACRO F1: 0.5851405696426532

              precision    recall  f1-score   support

           0       0.46      0.90      0.61      4708
           1       0.76      0.52      0.62      2117
           2       0.82      0.64      0.72      2232
           3       0.77      0.34      0.47      1996
           4       0.84      0.75      0.79      1715
           5       0.72      0.23      0.35      2611
           6       0.70      0.43      0.53       621

    accuracy                           0.60     16000
   macro avg       0.72      0.55      0.59     16000
weighted avg       0.68      0.60      0.58     16000



In [55]:
BANNED_ANCHOR_CLASSES = {0, 5}  # International, General

y_pred = []

for i, idx in enumerate(te):
	p = proba[i]
	text = X_text.iloc[idx]

	# modello sicuro → ok
	if p.max() >= tau:
		y_pred.append(p.argmax())
		continue

	# modello incerto → guarda top-2
	top2 = p.argsort()[-2:][::-1]

	toks = set(tokenize(text))
	override = None

	for c in top2:
		if c in BANNED_ANCHOR_CLASSES:
			continue
		if c in ANCHOR_WORDS and toks & ANCHOR_WORDS[c]:
			override = c
			break

	if override is not None:
		y_pred.append(override)
	else:
		y_pred.append(p.argmax())  # NON fallback globale


In [56]:
import pandas as pd
import numpy as np

# =========================
# LOAD RAW DATA
# =========================
df = pd.read_csv("../data/raw/development.csv")

# timestamp parsing
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# rimuoviamo i NaT solo per analisi (NON per training)
df_time = df.dropna(subset=["timestamp"]).copy()

print("Totale righe:", len(df))
print("Con timestamp valido:", len(df_time))
print("Percentuale con timestamp:", len(df_time) / len(df))
df_time["year"] = df_time["timestamp"].dt.year
df_time["month"] = df_time["timestamp"].dt.month
df_time["day"] = df_time["timestamp"].dt.day
df_time["weekday"] = df_time["timestamp"].dt.weekday  # 0=lun
df_time["hour"] = df_time["timestamp"].dt.hour
def hour_bucket(h):
	if 0 <= h < 6:
		return "night"
	elif 6 <= h < 12:
		return "morning"
	elif 12 <= h < 18:
		return "afternoon"
	else:
		return "evening"

df_time["hour_bucket"] = df_time["hour"].apply(hour_bucket)

print("\nCOUNTS BY CLASS")
print(df_time["label"].value_counts())
year_dist = pd.crosstab(df_time["year"], df_time["label"], normalize="columns")
print("\nYEAR DISTRIBUTION (per class)")
print(year_dist)
month_dist = pd.crosstab(df_time["month"], df_time["label"], normalize="columns")
print("\nMONTH DISTRIBUTION (per class)")
print(month_dist)
weekday_dist = pd.crosstab(df_time["weekday"], df_time["label"], normalize="columns")
print("\nWEEKDAY DISTRIBUTION (per class)")
print(weekday_dist)
hour_dist = pd.crosstab(df_time["hour"], df_time["label"], normalize="columns")
print("\nHOUR DISTRIBUTION (per class)")
print(hour_dist)
bucket_dist = pd.crosstab(df_time["hour_bucket"], df_time["label"], normalize="columns")
print("\nHOUR BUCKET DISTRIBUTION (per class)")
print(bucket_dist)
summary = (
	df_time
	.groupby("label")[["hour", "weekday", "month"]]
	.agg(["mean", "std"])
)

print("\nSUMMARY STATS")
print(summary)
from sklearn.feature_selection import mutual_info_classif

X_time = df_time[["hour", "weekday", "month"]]
y_time = df_time["label"]

mi = mutual_info_classif(X_time, y_time, discrete_features=True)

for col, val in zip(X_time.columns, mi):
	print(f"MI({col}, label) = {val:.4f}")


Totale righe: 79997
Con timestamp valido: 52247
Percentuale con timestamp: 0.6531119916996887

COUNTS BY CLASS
label
0    15406
2     8698
5     8135
1     7302
3     5934
4     4846
6     1926
Name: count, dtype: int64

YEAR DISTRIBUTION (per class)
label         0         1         2         3         4         5         6
year                                                                       
2004   0.171102  0.193509  0.104277  0.315639  0.346678  0.273755  0.263240
2005   0.035376  0.024651  0.010807  0.026626  0.034049  0.027044  0.021288
2006   0.211346  0.180635  0.199816  0.142905  0.160338  0.135710  0.142264
2007   0.443788  0.444810  0.543803  0.357937  0.326248  0.438476  0.399792
2008   0.138388  0.156396  0.141297  0.156892  0.132687  0.125015  0.173416

MONTH DISTRIBUTION (per class)
label         0         1         2         3         4         5         6
month                                                                      
1      0.119564  0.125171  0.1117